# Toy TFT on real outage windows + real GEE weather

This notebook:
- loads `POUS.csv`
- selects the same 10 Florida events used in the GEE export
- extracts outage windows from `timeseries.pq`
- loads the exported ERA5 hourly weather CSV
- joins outage and weather on `geoid` + `datetime`
- trains a tiny TFT and compares it with a baseline

This is a plumbing test, not a final model.


In [1]:
# Paths
POUS_PATH = "C:/Users/teaching/Downloads/outage-recovery-forecasting/data_raw/POUS.csv"
TIMESERIES_PATH = "C:/Users/teaching/Downloads/outage-recovery-forecasting/data_raw/timeseries.pq"
WEATHER_PATH = "C:/Users/teaching/Downloads/outage-recovery-forecasting/data_raw/toy_tft_florida_event_weather_era5_hourly.csv"


In [2]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error

import lightning.pytorch as pl
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss


In [3]:
# Event list used in the GEE script
# Keep GEOIDs as zero-padded strings.
events = pd.DataFrame([
    {"event_id": "event_001", "geoid": "12075", "event_start": "2017-09-09 22:00", "duration_hours": 190, "storm": "2017242N16333"},
    {"event_id": "event_002", "geoid": "12067", "event_start": "2017-09-09 20:00", "duration_hours": 191, "storm": "2017242N16333"},
    {"event_id": "event_003", "geoid": "12079", "event_start": "2017-09-10 02:00", "duration_hours": 124, "storm": "2017242N16333"},
    {"event_id": "event_004", "geoid": "12011", "event_start": "2017-09-10 06:00", "duration_hours": 178, "storm": "2017242N16333"},
    {"event_id": "event_005", "geoid": "12086", "event_start": "2017-09-10 06:00", "duration_hours": 210, "storm": "2017242N16333"},
    {"event_id": "event_006", "geoid": "12125", "event_start": "2017-09-10 11:00", "duration_hours": 133, "storm": "2017242N16333"},
    {"event_id": "event_007", "geoid": "12111", "event_start": "2017-09-10 14:00", "duration_hours": 131, "storm": "2017242N16333"},
    {"event_id": "event_008", "geoid": "12099", "event_start": "2017-09-10 14:00", "duration_hours": 147, "storm": "2017242N16333"},
    {"event_id": "event_009", "geoid": "12021", "event_start": "2017-09-10 14:00", "duration_hours": 263, "storm": "2017242N16333"},
    {"event_id": "event_010", "geoid": "12093", "event_start": "2017-09-10 16:00", "duration_hours": 175, "storm": "2017242N16333"},
])

events["event_start"] = pd.to_datetime(events["event_start"])
events


,event_id,geoid,event_start,duration_hours,storm
0,event_001,12075,2017-09-09 22:00:00,190,2017242N16333
1,event_002,12067,2017-09-09 20:00:00,191,2017242N16333
2,event_003,12079,2017-09-10 02:00:00,124,2017242N16333
3,event_004,12011,2017-09-10 06:00:00,178,2017242N16333
4,event_005,12086,2017-09-10 06:00:00,210,2017242N16333
5,event_006,12125,2017-09-10 11:00:00,133,2017242N16333
6,event_007,12111,2017-09-10 14:00:00,131,2017242N16333
7,event_008,12099,2017-09-10 14:00:00,147,2017242N16333
8,event_009,12021,2017-09-10 14:00:00,263,2017242N16333
9,event_010,12093,2017-09-10 16:00:00,175,2017242N16333


In [4]:
# Optional consistency check against POUS.csv
pous = pd.read_csv(POUS_PATH, dtype={"CountyFIPS": str})
pous["CountyFIPS"] = pous["CountyFIPS"].str.zfill(5)
pous["event_start"] = pd.to_datetime(pous["event_start"])

check = events.merge(
    pous[["CountyFIPS", "event_start", "duration_hours", "storm"]],
    left_on=["geoid", "event_start", "duration_hours", "storm"],
    right_on=["CountyFIPS", "event_start", "duration_hours", "storm"],
    how="left",
    indicator=True,
)
check[["event_id", "geoid", "event_start", "duration_hours", "storm", "_merge"]]


,event_id,geoid,event_start,duration_hours,storm,_merge
0,event_001,12075,2017-09-09 22:00:00,190,2017242N16333,both
1,event_002,12067,2017-09-09 20:00:00,191,2017242N16333,both
2,event_003,12079,2017-09-10 02:00:00,124,2017242N16333,both
3,event_004,12011,2017-09-10 06:00:00,178,2017242N16333,both
4,event_005,12086,2017-09-10 06:00:00,210,2017242N16333,both
5,event_006,12125,2017-09-10 11:00:00,133,2017242N16333,both
6,event_007,12111,2017-09-10 14:00:00,131,2017242N16333,both
7,event_008,12099,2017-09-10 14:00:00,147,2017242N16333,both
8,event_009,12021,2017-09-10 14:00:00,263,2017242N16333,both
9,event_010,12093,2017-09-10 16:00:00,175,2017242N16333,both


In [5]:
# Load the large outage time series.
# The parquet has a MultiIndex: RecordDateTime, CountyFIPS.
raw_timeseries = pd.read_parquet(TIMESERIES_PATH)
print(raw_timeseries.shape)
print(type(raw_timeseries.index))
print(raw_timeseries.index.names)
raw_timeseries.head()


(159522635, 2)
<class 'pandas.core.indexes.multi.MultiIndex'>
['RecordDateTime', 'CountyFIPS']


OutageFraction  CustomersTracked
RecordDateTime CountyFIPS                                  
2017-01-01     01001                  0.0               0.0
               01003                  0.0               0.0
               01005                  0.0               0.0
               01007                  0.0               0.0
               01009                  0.0               0.0

In [ ]:
# Extract event-centered outage windows from the large table.
# Window: [event_start - 12h, event_start + duration_hours + 12h]

def extract_event_window(raw_ts, geoid, event_start, duration_hours, event_id, storm):
    geoid = str(geoid).zfill(5)
    start = pd.Timestamp(event_start) - pd.Timedelta(hours=12)
    end = pd.Timestamp(event_start) + pd.Timedelta(hours=int(duration_hours) + 12)

    county_ts = raw_ts.xs(geoid, level="CountyFIPS").loc[start:end].copy()
    county_ts = county_ts.reset_index()
    county_ts = county_ts.rename(columns={
        "RecordDateTime": "datetime",
        "OutageFraction": "outageFraction",
        "CustomersTracked": "customersTracked",
    })
    county_ts["geoid"] = geoid
    county_ts["event_id"] = event_id
    county_ts["storm"] = storm
    county_ts["event_start"] = pd.Timestamp(event_start)
    county_ts["duration_hours"] = int(duration_hours)
    county_ts["t_rel_hours"] = ((county_ts["datetime"] - pd.Timestamp(event_start)).dt.total_seconds() / 3600).astype(int)
    county_ts = county_ts.sort_values("datetime").reset_index(drop=True)
    county_ts["time_idx"] = np.arange(len(county_ts))
    return county_ts

outage_windows = []
for row in events.itertuples(index=False):
    tmp = extract_event_window(
        raw_timeseries,
        geoid=row.geoid,
        event_start=row.event_start,
        duration_hours=row.duration_hours,
        event_id=row.event_id,
        storm=row.storm,
    )
    outage_windows.append(tmp)

outage_df = pd.concat(outage_windows, ignore_index=True)
print(outage_df.shape)
outage_df.head()


In [ ]:
# Quick sanity plots for a few events
sample_events = outage_df["event_id"].drop_duplicates().tolist()[:3]
for event_id in sample_events:
    tmp = outage_df[outage_df["event_id"] == event_id]
    plt.figure(figsize=(8, 3))
    plt.plot(tmp["datetime"], tmp["outageFraction"])
    plt.title(f"OutageFraction: {event_id}")
    plt.xlabel("datetime")
    plt.ylabel("outageFraction")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# Load GEE weather export
weather = pd.read_csv(WEATHER_PATH, dtype={"geoid": str})
weather["geoid"] = weather["geoid"].str.zfill(5)
weather["datetime"] = pd.to_datetime(weather["datetime"])
weather["event_start"] = pd.to_datetime(weather["event_start"])

print(weather.shape)
weather.head()


In [ ]:
# Join outage windows to weather on event_id + geoid + datetime
# event_id is included to avoid accidental mixing if the same county appears in multiple events.
df = outage_df.merge(
    weather,
    on=["event_id", "geoid", "datetime"],
    how="inner",
    suffixes=("", "_wx")
)

# Keep a clean county column
if "county" not in df.columns and "county_wx" in df.columns:
    df["county"] = df["county_wx"]

keep_cols = [
    "event_id", "storm", "geoid", "county", "datetime", "event_start",
    "duration_hours", "t_rel_hours", "time_idx",
    "outageFraction", "customersTracked",
    "gust_mps", "wind_speed_mps", "precip_mm", "pressure_hpa", "temp_c"
]
df = df[keep_cols].copy()
df = df.sort_values(["event_id", "datetime"]).reset_index(drop=True)

print(df.shape)
print(df.isna().mean().sort_values(ascending=False).head(10))
df.head()


In [ ]:
# Minimal missing-value handling
# Drop rows with missing target, then fill weather gaps within each event.
df = df.dropna(subset=["outageFraction"]).copy()

for col in ["gust_mps", "wind_speed_mps", "precip_mm", "pressure_hpa", "temp_c"]:
    df[col] = df.groupby("event_id")[col].transform(lambda s: s.interpolate(limit_direction="both"))

for col in ["gust_mps", "wind_speed_mps", "precip_mm", "pressure_hpa", "temp_c"]:
    df[col] = df.groupby("event_id")[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())

print(df.shape)
print(df.isna().sum())


In [ ]:
# Add simple calendar / relative-time features
df["hour"] = df["datetime"].dt.hour.astype(int)
df["dayofweek"] = df["datetime"].dt.dayofweek.astype(int)
df["month"] = df["datetime"].dt.month.astype(int)

lengths = df.groupby("event_id").size().sort_values()
lengths


In [ ]:
# Choose encoder / prediction lengths conservatively for this toy run
max_encoder_length = 48
max_prediction_length = 12

valid_events = lengths[lengths >= (max_encoder_length + max_prediction_length)].index.tolist()
df_model = df[df["event_id"].isin(valid_events)].copy()
print("Events kept:", len(valid_events))
print(df_model.groupby("event_id").size())


In [ ]:
# Time split within each event.
# This avoids the 'unknown category' error from unseen event_ids in validation.

def split_event(group):
    cutoff = group["time_idx"].max() - max_prediction_length
    group = group.copy()
    group["is_train"] = group["time_idx"] <= cutoff
    return group

df_model = df_model.groupby("event_id", group_keys=False).apply(split_event).reset_index(drop=True)
train_df = df_model[df_model["is_train"]].copy()
val_df = df_model.copy()

print(train_df.shape, val_df.shape)


In [ ]:
# Build TimeSeriesDataSet
training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="outageFraction",
    group_ids=["event_id"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["event_id", "geoid", "storm"],
    time_varying_known_reals=["time_idx", "t_rel_hours", "hour", "dayofweek", "month"],
    time_varying_unknown_reals=[
        "outageFraction",
        "gust_mps",
        "wind_speed_mps",
        "precip_mm",
        "pressure_hpa",
        "temp_c",
    ],
    target_normalizer=GroupNormalizer(groups=["event_id"]),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    val_df,
    predict=True,
    stop_randomization=True,
)

train_loader = training.to_dataloader(train=True, batch_size=64, num_workers=0)
val_loader = validation.to_dataloader(train=False, batch_size=128, num_workers=0)


In [ ]:
# Baseline
baseline = Baseline()
baseline_pred = baseline.predict(val_loader, return_y=True)

baseline_output = baseline_pred.output.detach().cpu().numpy().reshape(-1)
baseline_y = baseline_pred.y[0].detach().cpu().numpy().reshape(-1)
mask = np.isfinite(baseline_output) & np.isfinite(baseline_y)
print("Baseline MAE:", mean_absolute_error(baseline_y[mask], baseline_output[mask]))


In [ ]:
# Tiny TFT
pl.seed_everything(42)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=8,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=3,
)

print("Model parameters:", tft.size())

trainer = pl.Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    logger=True,
)

trainer.fit(tft, train_loader, val_loader)


In [ ]:
# TFT predictions and rough MAE
pred = tft.predict(val_loader, return_y=True)

out = pred.output.detach().cpu().numpy()
y = pred.y[0].detach().cpu().numpy()

if out.ndim == 3:
    q_idx = out.shape[-1] // 2
    out_point = out[:, :, q_idx]
else:
    out_point = out

tft_output = out_point.reshape(-1)
tft_y = y.reshape(-1)
mask = np.isfinite(tft_output) & np.isfinite(tft_y)
print("TFT MAE:", mean_absolute_error(tft_y[mask], tft_output[mask]))


In [ ]:
# Plot predictions for a couple of events
raw_pred, x = tft.predict(val_loader, mode="raw", return_x=True)

for idx in range(min(2, len(raw_pred["prediction"]))):
    tft.plot_prediction(x, raw_pred, idx=idx, add_loss_to_title=True)
    plt.tight_layout()
    plt.show()
